# measured separation of Exo-MerCat ID matches
- ID-matched planets sorted by crossmatching distance vs the 50'' allowed separation
- exports to `report/figures/emc_coordinate_calibration.pdf`

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / 'crossmatching').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from report_plots import common, data


In [ ]:
cme, input_table, matched = data.emc_crossmatch(dedup_input=False)
id_matched_planets = matched[matched["match_type"] != "coordinates"]
id_matched = cme.remove_duplicates(id_matched_planets, "star_name", from_file=str(data.EMC_CSV_PATH))
id_matched.sort("crossmatching_angular_separation")
len(id_matched)


In [ ]:
from matplotlib.ticker import FixedFormatter, FixedLocator


fig, ax = plt.subplots(figsize=(5, 3))

x = np.array(range(len(id_matched)))

plt.xlim(-2,len(id_matched)+2)
plt.yscale("log")
plt.axhline(50, linestyle="--", color="tab:orange", label="Allowed separation")
tick_values = [1E-3,1E-2, 1E-1, 1, 10, 50]
ax.yaxis.set_major_locator(FixedLocator(tick_values))
ax.yaxis.set_major_formatter(FixedFormatter([f"{i}" for i in tick_values]))
ax.xaxis.set_major_formatter(FixedFormatter([]))
plt.tick_params(axis="x", which="both", bottom=False, top=False, labelbottom=False)

ax.set_xlim(-5,len(id_matched) + 2)
ax.set_ylim(0.00001,100)
ax.set_ylabel("Crossmatching Distance [arcsec]")
ax.set_xlabel("ordered by inceasing crossmatching distance")
ax.fill_between(x,id_matched["crossmatching_angular_separation"],y2=0, step="mid", color="#aaaaaa")
ax.step(x, id_matched["crossmatching_angular_separation"], where="mid", lw=2, label="Measured separation")

ax.yaxis.minorticks_off()
ax.legend(loc="center left", bbox_to_anchor=(0,0.7))
ax.text(len(id_matched) / 2, 0.0002, "Below default\n100%", ha="center", va="center", fontsize=10)

common.save_figure("emc_coordinate_calibration")